# Experiment 7: CI/CD Pipeline for ML (Airbnb Price Prediction)

Run the cells in order in Google Colab.

## Cell 1: Install Dependencies

In [ ]:
!pip install -q dvc pytest flake8 mlflow pandas scikit-learn numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8

## Cell 2: Repository & Directory Setup

In [ ]:
import os

os.makedirs('src', exist_ok=True)
os.makedirs('tests', exist_ok=True)
os.makedirs('.github/workflows', exist_ok=True)

# Empty conftest.py so pytest adds the project root to sys.path (lets tests import `src.pipeline`)
open('conftest.py', 'w').close()

!git init -q
!dvc init --no-scm -q

## Cell 3: ML Pipeline Module (`src/pipeline.py`)

In [ ]:
pipeline_code = """import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def create_synthetic_airbnb_data(n_samples=500):
    np.random.seed(42)
    data = pd.DataFrame({
        'accommodates': np.random.randint(1, 10, n_samples),
        'bedrooms': np.random.randint(1, 5, n_samples),
        'beds': np.random.randint(1, 6, n_samples),
        'number_of_reviews': np.random.randint(0, 300, n_samples),
        'availability_365': np.random.randint(0, 365, n_samples),
    })
    data['price'] = (
        data['accommodates'] * 35 +
        data['bedrooms'] * 45 +
        data['beds'] * 20 +
        np.random.normal(0, 15, n_samples)
    ).clip(lower=30)
    return data

def train_model(df):
    X = df[['accommodates', 'bedrooms', 'beds', 'number_of_reviews', 'availability_365']]
    y = df['price']

    model = RandomForestRegressor(n_estimators=50, random_state=42)
    model.fit(X, y)
    preds = model.predict(X)

    mae = mean_absolute_error(y, preds)
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2 = r2_score(y, preds)

    return model, {'mae': mae, 'rmse': rmse, 'r2': r2}
"""

with open('src/pipeline.py', 'w') as f:
    f.write(pipeline_code)

## Cell 4: Test Suite (`tests/test_pipeline.py`)

In [ ]:
test_code = """import pytest
import pandas as pd
from src.pipeline import create_synthetic_airbnb_data, train_model

def test_dataset_generation():
    df = create_synthetic_airbnb_data(100)
    assert len(df) == 100
    assert 'price' in df.columns
    assert (df['price'] >= 0).all()

def test_model_training_performance():
    df = create_synthetic_airbnb_data(200)
    model, metrics = train_model(df)

    assert metrics['mae'] >= 0
    assert metrics['rmse'] >= 0
    assert metrics['r2'] >= 0.85
"""

with open('tests/test_pipeline.py', 'w') as f:
    f.write(test_code)

## Cell 5: Linting & Unit Testing Execution

In [ ]:
!flake8 src/ --count --select=E9,F63,F7,F82 --show-source --statistics
!pytest tests/test_pipeline.py -v

0
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: hydra-core-1.3.7, langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 2 items                                                              

tests/test_pipeline.py::test_dataset_generation PASSED                   [ 50%]
tests/test_pipeline.py::test_model_training_performance PASSED           [100%]

============================== 2 passed in 2.37s ===============================


## Cell 6: GitHub Actions Workflow (`.github/workflows/ci_cd.yml`)

In [ ]:
workflow_yaml = """name: Airbnb Price Prediction CI/CD Pipeline

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  lint:
    name: Code Quality & Linting (Flake8)
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Code
        uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'

      - name: Install Linting Tools
        run: |
          python -m pip install --upgrade pip
          pip install flake8

      - name: Run Flake8
        run: |
          flake8 src/ --count --select=E9,F63,F7,F82 --show-source --statistics

  test:
    name: Automated Unit Testing (pytest & DVC)
    needs: lint
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Code
        uses: actions/checkout@v3

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.10'

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pytest dvc pandas scikit-learn mlflow

      - name: Pull Data with DVC
        run: |
          dvc pull || echo "DVC pull skipped"

      - name: Run Pytest
        run: |
          pytest tests/ -v

  deploy:
    name: Model Build & Deployment
    needs: test
    runs-on: ubuntu-latest
    steps:
      - name: Deploy Model Artifact
        run: |
          echo "CI/CD Pipeline checks passed. Ready for deployment."
"""

with open('.github/workflows/ci_cd.yml', 'w') as f:
    f.write(workflow_yaml)